In [ ]:
from inference import generate_video
from cogvideox.transformer import CogVideoXTransformer3DModel
from cogvideox.scheduler import CogVideoXSwinDPMScheduler
from cogvideox.pipelines.pipeline_cogvideox import CogVideoXStreamingPipeline
import os
import torch
import json

In [ ]:
def test_checkpoint(
    model_path,
    transformer_path,
    output_dir,
    samples,
    dtype=torch.bfloat16,
    device="cuda:0",
    uncond_transformer_path=None,
    **kwargs
):
    transformer = CogVideoXTransformer3DModel.from_pretrained(
        os.path.join(transformer_path or model_path, "transformer"),
        torch_dtype=dtype,
        low_cpu_mem_usage=False,
    )

    if uncond_transformer_path is not None:
        uncond_transformer = CogVideoXTransformer3DModel.from_pretrained(
            os.path.join(uncond_transformer_path or model_path, "transformer"),
            torch_dtype=dtype,
            low_cpu_mem_usage=False,
        )
    else:
        uncond_transformer = None

    scheduler = CogVideoXSwinDPMScheduler.from_config(
        os.path.join(model_path, "scheduler"), timestep_spacing="trailing"
    )

    pipe = CogVideoXStreamingPipeline.from_pretrained(
        model_path,
        transformer=transformer,
        scheduler=scheduler,
        torch_dtype=dtype,
        uncond_transformer=uncond_transformer,
    )

    pipe.to(device)
    # pipe.enable_sequential_cpu_offload()
    pipe.vae.enable_slicing()
    pipe.vae.enable_tiling()

    os.makedirs(output_dir, exist_ok=True)

    for sample in samples:
        output_path = os.path.join(output_dir, sample["path"])
        if os.path.exists(output_path):
            continue
        generate_video(
            prompt=sample["caption"],
            model_path=model_path,
            transformer_path=transformer_path,
            output_path=os.path.join(output_dir, sample["path"]),
            video_path=sample["video_path"],
            control_signal=",".join(sample["control_codes"]),
            pipe=pipe,
            **kwargs
        )

In [ ]:
import glob
import random
import json

In [ ]:
sample_paths = glob.glob('/mbz/users/yi.gu/yichi/game_datasets/outputs/matrix_forza_horizon_5_eval_set/*.json')
sample_paths.sort()

rng = random.Random(42)
sample_paths = rng.sample(sample_paths, 10)

samples = []
for path in sample_paths:
    with open(path, 'r') as file:
        sample = json.load(file)
        sample['video_path'] = path.removesuffix('.json') + '.mp4'
        samples.append(sample)

In [ ]:
samples[0]

In [ ]:
samples_no_env = [{**sample, "caption": None} for sample in samples]

In [ ]:
samples_no_env[0]

In [ ]:
default_args = {
    "guidance_scale": 6.0,
    "num_inference_steps": 20,
    "width": 720,
    "height": 480,
    "seed": 42,
    "num_noise_groups": 4,
    "control_signal_type": "raw",
    "show_progress": "outer",
}

In [ ]:
stage3 = "/mbz/users/yi.gu/.cache/huggingface/hub/models--MatrixTeam--TheMatrix/snapshots/06364bfc591b2a4ef18c10aa33904117e9974f2c/stage3"

In [15]:
test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise/checkpoints/checkpoint-2932',
    output_dir='eval_results/swin_dpm_forza_horizon_with_env_cond_noise-2932-test',
    samples=samples[:3],
    fps=20,
    num_sample_groups=128,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    **default_args
)

Loading pipeline components...: 100%|██████████| 5/5 [00:00<00:00,  5.43it/s]


In [ ]:
test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise/checkpoints/checkpoint-2932',
    output_dir='eval_results/swin_dpm_forza_horizon_with_env_cond_noise-2932',
    samples=samples,
    fps=20,
    num_sample_groups=128,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    **default_args
)

In [ ]:
test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise_from_cogv/checkpoints/checkpoint-2932',
    output_dir='eval_results/swin_dpm_forza_horizon_with_env_cond_noise_from_cogv-2932',
    samples=samples,
    fps=20,
    num_sample_groups=128,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    **default_args
)

In [ ]:
test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise_from_cogv/checkpoints/checkpoint-8796',
    output_dir='eval_results/swin_dpm_forza_horizon_with_env_cond_noise_from_cogv-8796',
    samples=samples[:3],
    fps=20,
    num_sample_groups=128,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    **default_args
)

In [ ]:
test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_no_env_cond_noise/checkpoints/checkpoint-2932',
    output_dir='eval_results/swin_dpm_forza_horizon_no_env_cond_noise-2932',
    samples=samples_no_env[:3],
    fps=20,
    num_sample_groups=128,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    **default_args
)

In [ ]:
test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/matrix/stage3/output/sft/stage3/2025-03-26_15-18-43/checkpoint-3000',
    output_dir='eval_results/matrix-3000',
    samples=samples[:3],
    fps=16,
    num_sample_groups=128,
    init_video_clip_frame=65,
    actions_in_prompt=True,
    actions_in_prompt_repeat=1,
    **default_args
)

In [ ]:
test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/matrix/stage3/output/sft/stage3/2025-03-26_15-18-43/checkpoint-1000',
    output_dir='eval_results/matrix-1000',
    samples=samples[:3],
    fps=16,
    num_sample_groups=128,
    init_video_clip_frame=65,
    actions_in_prompt=True,
    actions_in_prompt_repeat=1,
    **default_args
)

In [ ]:
test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise_empty_string/checkpoints/checkpoint-2932',
    output_dir='eval_results/swin_dpm_forza_horizon_with_env_cond_noise_empty_string-2932',
    samples=samples,
    fps=20,
    num_sample_groups=128,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    **default_args
)

In [ ]:
test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise_empty_string/checkpoints/checkpoint-14660',
    output_dir='eval_results/swin_dpm_forza_horizon_with_env_cond_noise_empty_string-14660',
    samples=samples[:5],
    fps=20,
    num_sample_groups=128,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    **default_args
)

In [ ]:
test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise_empty_string_fixed/checkpoints/checkpoint-2932',
    output_dir='eval_results/swin_dpm_forza_horizon_with_env_cond_noise_empty_string_fixed-2932',
    samples=samples[:3],
    fps=20,
    num_sample_groups=128,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    **default_args
)

In [ ]:
test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise/checkpoints/checkpoint-2932',
    output_dir='eval_results/swin_dpm_forza_horizon_with_env_cond_noise-2932-long',
    samples=[s for s in samples if s['path'] == '2024-07-04_00-52-12_000026_segment_start492_end922.mp4'],
    fps=20,
    num_sample_groups=1280,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    **default_args
)

In [ ]:
mario_samle = {
    "video_path": "/mbz/users/yi.gu/yichi/world-model-hf-trainer/world_1-1_bootstrap_video.mp4",
    "path": "world_1-1_bootstrap_video.mp4",
    "control_codes": ["N"] * 33,
    "caption": "Blue sky, a white cloud, bottom brick platform, bottom green bush.",
}


def mario_actions_in_prompt(controls, caption):
    def get_action(code):
        mapping = {"N": "none", "D": "right A B"}
        return mapping.get(code, "right B")

    actions = [get_action(c) for c in controls]
    chunk_size = len(actions) // 4
    chunks = [actions[i : i + chunk_size] for i in range(0, len(actions), chunk_size)]
    chunks = [chunk[:8] for chunk in chunks]  # 8 actions per chunk
    chunks = ['; '.join(chunk) for chunk in chunks]
    prompt = ' | '.join(f'{chunk} / {caption}' for chunk in chunks)
    return prompt

In [ ]:
args = default_args.copy()
args.pop('num_inference_steps')
# args.pop('guidance_scale')

test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yiyan/world-model-hf-trainer/output/swin_dpm_mario_pad_53k/checkpoints/checkpoint-9344',
    output_dir='eval_results/swin_dpm_mario_pad_53k-9344-cfg',
    samples=[mario_samle],
    fps=8,
    num_sample_groups=32,
    init_video_clip_frame=33,
    actions_in_prompt=True,
    actions_in_prompt_fn=mario_actions_in_prompt,
    resize_mode='pad',
    # guidance_scale=0.0,
    num_inference_steps=48,
    **args,
)

In [ ]:
args = default_args.copy()
# args.pop('num_inference_steps')
args.pop('guidance_scale')

test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise_empty_string/checkpoints/checkpoint-14660',
    output_dir='eval_results/swin_dpm_forza_horizon_with_env_cond_noise_empty_string-14660-uncond',
    samples=samples[:5],
    fps=20,
    num_sample_groups=32,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    actions_in_prompt_fn=lambda x, y: "",
    guidance_scale=0.0,
    **args
)

In [ ]:
args = default_args.copy()
# args.pop('num_inference_steps')
args.pop('guidance_scale')

test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise_empty_string_20/checkpoints/checkpoint-10262',
    output_dir='eval_results/swin_dpm_forza_horizon_with_env_cond_noise_empty_string_20-10262-uncond',
    samples=samples[:5],
    fps=20,
    num_sample_groups=32,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    actions_in_prompt_fn=lambda x, y: "",
    guidance_scale=0.0,
    **args
)

In [ ]:
test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise_empty_string_20/checkpoints/checkpoint-10262',
    output_dir='eval_results/swin_dpm_forza_horizon_with_env_cond_noise_empty_string_20-10262',
    samples=samples[:5],
    fps=20,
    num_sample_groups=128,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    **default_args
)

In [ ]:

args = default_args.copy()
args.pop('guidance_scale')

test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise_empty_string_20/checkpoints/checkpoint-10262',
    output_dir='eval_results/swin_dpm_forza_horizon_with_env_cond_noise_empty_string_20-10262-gs2',
    samples=samples[:5],
    fps=20,
    num_sample_groups=128,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    guidance_scale=2.0,
    **args
)

In [ ]:
args = default_args.copy()
args.pop('guidance_scale')

test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise_empty_string_20/checkpoints/checkpoint-10262',
    output_dir='eval_results/swin_dpm_forza_horizon_with_env_cond_noise_empty_string_20-10262-gs4',
    samples=samples[:5],
    fps=20,
    num_sample_groups=128,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    guidance_scale=4.0,
    **args
)

In [ ]:
args = default_args.copy()
args.pop('guidance_scale')

test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise_empty_string_20/checkpoints/checkpoint-10262',
    output_dir='eval_results/swin_dpm_forza_horizon_with_env_cond_noise_empty_string_20-10262-gs8',
    samples=samples[:5],
    fps=20,
    num_sample_groups=128,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    guidance_scale=8.0,
    **args
)

In [ ]:
args = default_args.copy()
# args.pop('num_inference_steps')
args.pop('guidance_scale')

test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise/checkpoints/checkpoint-2932',
    output_dir='eval_results/swin_dpm_forza_horizon_with_env_cond_noise-2932-uncond',
    samples=samples[:5],
    fps=20,
    num_sample_groups=32,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    actions_in_prompt_fn=lambda x, y: "",
    guidance_scale=0.0,
    **args
)

In [12]:
test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise/checkpoints/checkpoint-2932',
    uncond_transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise_empty_string_20/checkpoints/checkpoint-10262',
    output_dir='eval_results/with_env_cond_noise-2932-uncond-empty_string_20-10262',
    samples=samples[:5],
    fps=20,
    num_sample_groups=128,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    **default_args
)

/mbz/users/yi.gu/.conda/envs/wm2-yichi/lib/python3.10/site-packages/diffusers/configuration_utils.py:245: FutureWarning: It is deprecated to pass a pretrained model name or path to `from_config`.If you were trying to load a scheduler, please use <class 'cogvideox.scheduler.CogVideoXSwinDPMScheduler'>.from_pretrained(...) instead. Otherwise, please make sure to pass a configuration dictionary instead. This functionality will be removed in v1.0.0.
  deprecate("config-passed-as-path", "1.0.0", deprecation_message, standard_warn=False)
Loading pipeline components...: 100%|██████████| 5/5 [00:00<00:00,  6.17it/s]


Padded control signal: D,D,DL,DL,DL,DL,DL,DL,DL,D,D,D,D,D,D,D,D,D,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,DR,D,D,D,D,DL,DL,DL,DL,D,D,D,D,D,DL,DL,DL,D,D,DL,DL,DL,DL,D,D,D,D,DL,DL,D,D,D,DL,DL,DL,DL,DL,D,D,D,D,DL,DL,DL,DL,D,D,DL,DL,DL,DL,DL,D,D,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,D,D,D,DL,DL,DL,D,D,D,D,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,D,D,D,D,DL,DL,D,D,D,D,D,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,D,D,D,D,DL,DL,DL,DL,DL,D,D,D,D,D,DL,DL,D,D,D,DR,DR,D,D,DR,DR,D,D,D,D,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,DL,DL,DL,D,D,D,D,D,DL,DL,DL,D,D,D,DL,DL,DL,DL,D,D,D,DL,DL,DL,DL,D,D,DL,DL,DL,DL,DL,D,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,DR,DR,DR,D,D,DR,DR,DR,DR,D,D,D,D,D,DR,DR,D,D,D,DR,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,D,D,D,DL,DL,DL,D,D,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,D,D,DL,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,DR,D,D,DR,DR,D,D,D,DL,DL,D,D,D,DR,DR,D,D,DR,DR,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,D,DR,DR,DR,DR,D,D,D,

100%|██████████| 128/128 [09:03<00:00,  4.25s/it]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Padded control signal: D,D,D,D,D,D,DR,DR,DR,DR,DR,DR,DR,D,D,D,D,D,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,DR,D,D,D,D,DL,DL,DL,DL,D,D,D,D,D,DL,DL,DL,D,D,DL,DL,DL,DL,D,D,D,D,DL,DL,D,D,D,DL,DL,DL,DL,DL,D,D,D,D,DL,DL,DL,DL,D,D,DL,DL,DL,DL,DL,D,D,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,D,D,D,DL,DL,DL,D,D,D,D,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,D,D,D,D,DL,DL,D,D,D,D,D,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,D,D,D,D,DL,DL,DL,DL,DL,D,D,D,D,D,DL,DL,D,D,D,DR,DR,D,D,DR,DR,D,D,D,D,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,DL,DL,DL,D,D,D,D,D,DL,DL,DL,D,D,D,DL,DL,DL,DL,D,D,D,DL,DL,DL,DL,D,D,DL,DL,DL,DL,DL,D,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,DR,DR,DR,D,D,DR,DR,DR,DR,D,D,D,D,D,DR,DR,D,D,D,DR,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,D,D,D,DL,DL,DL,D,D,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,D,D,DL,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,DR,D,D,DR,DR,D,D,D,DL,DL,D,D,D,DR,DR,D,D,DR,DR,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,D,DR,DR,DR,DR,D,D,D,

  0%|          | 0/128 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (241 > 226). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because `max_sequence_length` is set to  226 tokens: ['ing path. The sky is tinged with shades of blue.']
100%|██████████| 128/128 [09:02<00:00,  4.24s/it]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Padded control signal: DR,D,D,D,D,D,D,D,D,D,DR,DR,DR,D,D,D,D,D,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,DR,D,D,D,D,DL,DL,DL,DL,D,D,D,D,D,DL,DL,DL,D,D,DL,DL,DL,DL,D,D,D,D,DL,DL,D,D,D,DL,DL,DL,DL,DL,D,D,D,D,DL,DL,DL,DL,D,D,DL,DL,DL,DL,DL,D,D,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,D,D,D,DL,DL,DL,D,D,D,D,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,D,D,D,D,DL,DL,D,D,D,D,D,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,D,D,D,D,DL,DL,DL,DL,DL,D,D,D,D,D,DL,DL,D,D,D,DR,DR,D,D,DR,DR,D,D,D,D,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,DL,DL,DL,D,D,D,D,D,DL,DL,DL,D,D,D,DL,DL,DL,DL,D,D,D,DL,DL,DL,DL,D,D,DL,DL,DL,DL,DL,D,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,DR,DR,DR,D,D,DR,DR,DR,DR,D,D,D,D,D,DR,DR,D,D,D,DR,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,D,D,D,DL,DL,DL,D,D,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,D,D,DL,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,DR,D,D,DR,DR,D,D,D,DL,DL,D,D,D,DR,DR,D,D,DR,DR,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,D,DR,DR,DR,DR,D,D,D,DR,

 18%|█▊        | 23/128 [01:38<07:31,  4.30s/it]


KeyboardInterrupt: 

In [13]:
test_checkpoint(
    model_path=stage3,
    transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise_empty_string_20/checkpoints/checkpoint-10262',
    uncond_transformer_path='/mbz/users/yi.gu/yichi/world-model-hf-trainer/output/swin_dpm_forza_horizon_with_env_cond_noise/checkpoints/checkpoint-2932',
    output_dir='eval_results/empty_string_20-10262-uncond-with_env_cond_noise-2932',
    samples=samples[:5],
    fps=20,
    num_sample_groups=128,
    init_video_clip_frame=49,
    actions_in_prompt=True,
    **default_args
)

Loading pipeline components...: 100%|██████████| 5/5 [00:00<00:00,  5.04it/s]


Padded control signal: D,D,DL,DL,DL,DL,DL,DL,DL,D,D,D,D,D,D,D,D,D,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,DR,D,D,D,D,DL,DL,DL,DL,D,D,D,D,D,DL,DL,DL,D,D,DL,DL,DL,DL,D,D,D,D,DL,DL,D,D,D,DL,DL,DL,DL,DL,D,D,D,D,DL,DL,DL,DL,D,D,DL,DL,DL,DL,DL,D,D,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,D,D,D,DL,DL,DL,D,D,D,D,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,D,D,D,D,DL,DL,D,D,D,D,D,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,D,D,D,D,DL,DL,DL,DL,DL,D,D,D,D,D,DL,DL,D,D,D,DR,DR,D,D,DR,DR,D,D,D,D,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,DL,DL,DL,D,D,D,D,D,DL,DL,DL,D,D,D,DL,DL,DL,DL,D,D,D,DL,DL,DL,DL,D,D,DL,DL,DL,DL,DL,D,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,DR,DR,DR,D,D,DR,DR,DR,DR,D,D,D,D,D,DR,DR,D,D,D,DR,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,D,D,D,DL,DL,DL,D,D,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,D,D,DL,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,DR,D,D,DR,DR,D,D,D,DL,DL,D,D,D,DR,DR,D,D,DR,DR,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,D,DR,DR,DR,DR,D,D,D,

100%|██████████| 128/128 [09:05<00:00,  4.26s/it]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Padded control signal: D,D,D,D,D,D,DR,DR,DR,DR,DR,DR,DR,D,D,D,D,D,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,DR,D,D,D,D,DL,DL,DL,DL,D,D,D,D,D,DL,DL,DL,D,D,DL,DL,DL,DL,D,D,D,D,DL,DL,D,D,D,DL,DL,DL,DL,DL,D,D,D,D,DL,DL,DL,DL,D,D,DL,DL,DL,DL,DL,D,D,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,D,D,D,DL,DL,DL,D,D,D,D,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,D,D,D,D,DL,DL,D,D,D,D,D,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,D,D,D,D,DL,DL,DL,DL,DL,D,D,D,D,D,DL,DL,D,D,D,DR,DR,D,D,DR,DR,D,D,D,D,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,DL,DL,DL,D,D,D,D,D,DL,DL,DL,D,D,D,DL,DL,DL,DL,D,D,D,DL,DL,DL,DL,D,D,DL,DL,DL,DL,DL,D,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,DR,DR,DR,D,D,DR,DR,DR,DR,D,D,D,D,D,DR,DR,D,D,D,DR,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,D,D,D,DL,DL,DL,D,D,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,D,D,DL,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,DR,D,D,DR,DR,D,D,D,DL,DL,D,D,D,DR,DR,D,D,DR,DR,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,D,DR,DR,DR,DR,D,D,D,

  0%|          | 0/128 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (241 > 226). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because `max_sequence_length` is set to  226 tokens: ['ing path. The sky is tinged with shades of blue.']
100%|██████████| 128/128 [09:04<00:00,  4.25s/it]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


...

100%|██████████| 128/128 [09:04<00:00,  4.25s/it]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Padded control signal: D,D,D,D,D,D,D,D,D,DR,DR,DR,DR,D,D,D,D,D,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,DR,D,D,D,D,DL,DL,DL,DL,D,D,D,D,D,DL,DL,DL,D,D,DL,DL,DL,DL,D,D,D,D,DL,DL,D,D,D,DL,DL,DL,DL,DL,D,D,D,D,DL,DL,DL,DL,D,D,DL,DL,DL,DL,DL,D,D,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,D,D,D,DL,DL,DL,D,D,D,D,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,D,D,D,D,DL,DL,D,D,D,D,D,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,D,D,D,D,DL,DL,DL,DL,DL,D,D,D,D,D,DL,DL,D,D,D,DR,DR,D,D,DR,DR,D,D,D,D,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,DL,DL,DL,D,D,D,D,D,DL,DL,DL,D,D,D,DL,DL,DL,DL,D,D,D,DL,DL,DL,DL,D,D,DL,DL,DL,DL,DL,D,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,DR,DR,DR,D,D,DR,DR,DR,DR,D,D,D,D,D,DR,DR,D,D,D,DR,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,D,D,D,DL,DL,DL,D,D,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,D,D,DL,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,DR,D,D,DR,DR,D,D,D,DL,DL,D,D,D,DR,DR,D,D,DR,DR,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,D,DR,DR,DR,DR,D,D,D,DR,

100%|██████████| 128/128 [09:04<00:00,  4.25s/it]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Padded control signal: D,D,D,D,D,D,D,DR,DR,DR,DR,DR,DR,D,D,D,D,D,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,DR,D,D,D,D,DL,DL,DL,DL,D,D,D,D,D,DL,DL,DL,D,D,DL,DL,DL,DL,D,D,D,D,DL,DL,D,D,D,DL,DL,DL,DL,DL,D,D,D,D,DL,DL,DL,DL,D,D,DL,DL,DL,DL,DL,D,D,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,D,D,D,DL,DL,DL,D,D,D,D,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,D,D,D,D,DL,DL,D,D,D,D,D,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,D,D,D,D,DL,DL,DL,DL,DL,D,D,D,D,D,DL,DL,D,D,D,DR,DR,D,D,DR,DR,D,D,D,D,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,DL,DL,DL,D,D,D,D,D,DL,DL,DL,D,D,D,DL,DL,DL,DL,D,D,D,DL,DL,DL,DL,D,D,DL,DL,DL,DL,DL,D,D,D,D,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,DR,DR,DR,D,D,DR,DR,DR,DR,D,D,D,D,D,DR,DR,D,D,D,DR,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,D,D,D,DL,DL,DL,D,D,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,D,D,DL,DL,DL,DL,D,D,D,D,D,DR,DR,DR,DR,DR,DR,D,D,D,DR,DR,DR,DR,DR,D,D,DR,DR,D,D,D,DL,DL,D,D,D,DR,DR,D,D,DR,DR,D,D,DR,DR,D,D,D,D,DR,DR,DR,DR,DR,DR,D,D,DR,DR,DR,D,D,D,D,DR,DR,DR,DR,D,D,D,D

100%|██████████| 128/128 [09:04<00:00,  4.25s/it]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
